# Weather summary

We'll try to get a weather summary for the next week for a specific city.

## Load and check API key

In [ ]:
import os
from dotenv import load_dotenv


# Load environment variables in a file called .env
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


## Functions to get city coordinates and weather forecast.

In [ ]:
import requests
import json
from typing import TypedDict, Any


class Coordinates(TypedDict):
    latitude: float
    longitude: float


def get_city_coordinates(city: str) -> Coordinates | None:
    """
    Get the latitude and longitude of a city.
    """
    response = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1&language=en&format=json")
    response.raise_for_status()
    data = response.json()
    if "results" in data and (results := data["results"]):
        return results[0]
    return None


JSON = dict[str, Any]


def get_weather_data(coordinates: Coordinates) -> JSON:
    """
    Get the weather data for a specific city for the next week.
    """
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={coordinates['latitude']}&longitude={coordinates['longitude']}&daily=weathercode,temperature_2m_max,temperature_2m_min&timezone=auto")
    response.raise_for_status()
    return response.json()


## OpenAI interaction code

In [ ]:
import openai

system_prompt = """
You are a assistant that analyzes weather in a specific city for the next week,
and provides a short summary of the weather provided to you in a JSON format.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt = """
Here are the weather data in a JSON format.
Provide a short summary of the weather in a markdown format.

{weather_data}
"""


def format_json(data: JSON) -> str:
    """
    Format the weather data as a JSON string.
    """
    return json.dumps(data, indent=2)


def summarize_weather_data(weather_data: JSON) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt.format(weather_data=format_json(weather_data))}
    ]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

## Result formatting

In [ ]:
from IPython.display import DisplayHandle, Markdown, display

def display_summary(summary: str) -> DisplayHandle | None:
    return display(Markdown(summary))

## Put it all together

In [ ]:
def get_weather_summary(city: str) -> DisplayHandle | None:
    coordinates = get_city_coordinates(city)
    if coordinates is None:
        return display("Could not find coordinates for {city}")
    weather_data = get_weather_data(coordinates)
    summary = summarize_weather_data(weather_data)
    return display_summary(summary)


## Demo

In [ ]:
get_weather_summary("San Francisco")

In [ ]:
get_weather_summary("London")

In [ ]:
get_weather_summary("Nonexistingcity")